In [4]:
import pandas as pd
import re

# ── 1. Load raw file ──────────────────────────────────────────────────────────
RAW_PATH = "Average_retail_price_of_electricity.csv"   # adjust if needed
OUT_PATH  = "electricity_prices_tidy.csv"

raw = pd.read_csv(RAW_PATH, header=None, skiprows=4)

# Row 0 (after skip) is the header row: description, units, source key, Apr 2013, …
raw.columns = raw.iloc[0]
raw = raw.drop(index=raw.index[0]).reset_index(drop=True)

# ── 2. Keep only commercial and industrial rows ───────────────────────────────
mask = raw["description"].str.contains(r": commercial|: industrial", case=False, na=False)
df = raw[mask].copy()

# ── 3. State abbreviation lookup ─────────────────────────────────────────────
STATE_ABBR = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY",
    "District Of Columbia": "DC", "District of Columbia": "DC",
}

# ── 4. Parse state name and category from description ────────────────────────
def parse_row(desc):
    """Return (state_abbr, category) from e.g. 'Connecticut : commercial'."""
    parts = desc.split(" : ")
    state_name = parts[0].strip()
    category   = parts[1].strip().lower()   # 'commercial' or 'industrial'
    abbr = STATE_ABBR.get(state_name)
    return abbr, category

df[["state", "category"]] = df["description"].apply(
    lambda d: pd.Series(parse_row(d))
)

# Drop rows where state wasn't matched (e.g. regional aggregates)
df = df.dropna(subset=["state"])

# ── 5. Melt from wide to long ─────────────────────────────────────────────────
date_cols = [c for c in df.columns if re.match(r"[A-Za-z]+ \d{4}", str(c))]

long = df.melt(
    id_vars=["state", "category"],
    value_vars=date_cols,
    var_name="date_str",
    value_name="price",
)

# ── 6. Parse year / month and clean up ───────────────────────────────────────
long["date"] = pd.to_datetime(long["date_str"], format="%b %Y")
long["year"]  = long["date"].dt.year
long["month"] = long["date"].dt.month

long["price"] = pd.to_numeric(long["price"], errors="coerce")
long = long.dropna(subset=["price"])

# ── 7. Final column order and sort ───────────────────────────────────────────
result = (
    long[["year", "month", "state", "category", "price"]]
    .sort_values(["state", "category", "year", "month"])
    .reset_index(drop=True)
)

# ── 8. Save ───────────────────────────────────────────────────────────────────
result.to_csv(OUT_PATH, index=False)
print(f"Saved {len(result):,} rows → {OUT_PATH}")
print(result.head(10).to_string(index=False))

Saved 1,848 rows → electricity_prices_tidy.csv
 year  month state   category  price
 2013      4    CT commercial  14.58
 2013      5    CT commercial  14.55
 2013      6    CT commercial  14.68
 2013      7    CT commercial  14.36
 2013      8    CT commercial  14.28
 2013      9    CT commercial  14.48
 2013     10    CT commercial  14.63
 2013     11    CT commercial  14.73
 2013     12    CT commercial  14.73
 2014      1    CT commercial  16.50
